# Data Preparation

## 1. Select data

Data source: https://www.kaggle.com/datasets/sturarods/anac-national-civil-aviation-agency-2000-2025?select=anac_brazil.csv

Selected attriblutes with rationale:
- `nr_passag_pagos`: revenue passengers - primary target, the core metric required.
- `nr_ano_mes_partida_real`: date in format YYYYMM for monthly aggregation.
- `ds_grupo_di`: filters for regular flights, excludes not regular (charters) and not productive (ferry/training).
- `ds_servico_tipo_linha`: filters for passenger flights, excludes cargo ones.
- `ds_natureza_tipo_linha`: separates domestic and international flights.
- `nm_pais_origem` and `nm_pais_destino`: restore missing values or not identified records in `ds_natureza_tipo_linha`.

The other attributes should be excluded because they are either derived from attributes mentioned above or do not carry useful information regarding national monthly passengers forecast.

## 2. Clean data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np

chunk_size = 100000
chunks = []

reader = pd.read_csv(
    'anac_brazil.csv',
    usecols=['ds_grupo_di', 'ds_natureza_tipo_linha', 'nm_pais_origem', 'nm_pais_destino', 'ds_servico_tipo_linha', 'nr_passag_pagos', 'nr_ano_mes_partida_real'],
    chunksize=chunk_size
)

for chunk in reader:
    # filter regular flights
    chunk = chunk[chunk['ds_grupo_di'] == 'REGULAR'].copy()
    if chunk.empty:
        continue
    
    # identify domestic/international flights
    not_identified = chunk['ds_natureza_tipo_linha'] == 'NÃO IDENTIFICADO'
    international = (chunk['nm_pais_origem'] != 'BRASIL') | (chunk['nm_pais_destino'] != 'BRASIL')
    
    chunk['ds_natureza_tipo_linha'] = np.select(
        [
            not_identified & international, 
            not_identified & ~international
        ], 
        [
            'INTERNACIONAL', 
            'DOMÉSTICA'
        ], 
        default=chunk['ds_natureza_tipo_linha']
    )
    
    # filter passenger flights
    passenger_flight = (chunk['nr_passag_pagos'] > 0.0) & (chunk['nr_passag_pagos'] <= 853.0) & (chunk['ds_servico_tipo_linha'].isin(['PASSAGEIRO', 'NÃO IDENTIFICADO']))
    chunk = chunk[passenger_flight]
    
    chunks.append(chunk)

df_filtered = pd.concat(chunks, ignore_index=True)

In [3]:
df_clean = df_filtered[['nr_ano_mes_partida_real', 'ds_natureza_tipo_linha', 'nr_passag_pagos']]

In [4]:
# date field: drop nan values, convert to date type
df_clean = df_clean.dropna(subset=['nr_ano_mes_partida_real'])
df_clean['nr_ano_mes_partida_real'] = pd.to_datetime(df_clean["nr_ano_mes_partida_real"], errors="coerce", format='%Y%m')

In [5]:
# passengers filed: convert to int type
df_clean['nr_passag_pagos'] = df_clean['nr_passag_pagos'].astype(int)

In [6]:
# split dataset into two datasets with domestic and international flights
domestic_df = df_clean[df_clean['ds_natureza_tipo_linha'] == 'DOMÉSTICA']
international_df = df_clean[df_clean['ds_natureza_tipo_linha'] == 'INTERNACIONAL']

# drop unnecessary column ds_natureza_tipo_linha from both datasets
domestic_df = domestic_df.drop('ds_natureza_tipo_linha', axis=1)
international_df = international_df.drop('ds_natureza_tipo_linha', axis=1)

## 3. Construct data

### Derived attributes

- sliding window: capture short‑term momentum and seasonality.
- expanding window: capture long‑term trend information.


In [12]:
# group passengers by months and sort

domestic_monthly = (
    domestic_df
    .groupby('nr_ano_mes_partida_real')['nr_passag_pagos']
    .sum()
    .reset_index()
    .rename(columns={'nr_passag_pagos': 'passengers'})
)

international_monthly = (
    international_df
    .groupby('nr_ano_mes_partida_real')['nr_passag_pagos']
    .sum()
    .reset_index()
    .rename(columns={'nr_passag_pagos': 'passengers'})
)

domestic_monthly = domestic_monthly.sort_values('nr_ano_mes_partida_real').reset_index(drop=True)
international_monthly = international_monthly.sort_values('nr_ano_mes_partida_real').reset_index(drop=True)

In [14]:
# sliding window
domestic_monthly['lag_1']  = domestic_monthly['passengers'].shift(1)
domestic_monthly['lag_3']  = domestic_monthly['passengers'].shift(3)
domestic_monthly['lag_6']  = domestic_monthly['passengers'].shift(6)
domestic_monthly['lag_12'] = domestic_monthly['passengers'].shift(12)

international_monthly['lag_1']  = international_monthly['passengers'].shift(1)
international_monthly['lag_3']  = international_monthly['passengers'].shift(3)
international_monthly['lag_6']  = international_monthly['passengers'].shift(6)
international_monthly['lag_12'] = international_monthly['passengers'].shift(12)

# rolling means
domestic_monthly['roll_mean_3'] = domestic_monthly['passengers'].shift(1).rolling(window=3).mean()
domestic_monthly['roll_mean_6'] = domestic_monthly['passengers'].shift(1).rolling(window=6).mean()
domestic_monthly['roll_mean_12'] = domestic_monthly['passengers'].shift(1).rolling(window=12).mean()

international_monthly['roll_mean_3'] = international_monthly['passengers'].shift(1).rolling(window=3).mean()
international_monthly['roll_mean_6'] = international_monthly['passengers'].shift(1).rolling(window=6).mean()
international_monthly['roll_mean_12'] = international_monthly['passengers'].shift(1).rolling(window=12).mean()

# rolling standard deviation
domestic_monthly['roll_std_6'] = domestic_monthly['passengers'].shift(1).rolling(window=6).std()

international_monthly['roll_std_6'] = international_monthly['passengers'].shift(1).rolling(window=6).std()

# expanding window statistics
domestic_monthly['exp_mean'] = domestic_monthly['passengers'].shift(1).expanding().mean()
domestic_monthly['exp_std']  = domestic_monthly['passengers'].shift(1).expanding().std()
domestic_monthly['exp_min']  = domestic_monthly['passengers'].shift(1).expanding().min()
domestic_monthly['exp_max']  = domestic_monthly['passengers'].shift(1).expanding().max()

international_monthly['exp_mean'] = international_monthly['passengers'].shift(1).expanding().mean()
international_monthly['exp_std']  = international_monthly['passengers'].shift(1).expanding().std()
international_monthly['exp_min']  = international_monthly['passengers'].shift(1).expanding().min()
international_monthly['exp_max']  = international_monthly['passengers'].shift(1).expanding().max()

## 4. Integrate data

The is no other data to merge with.

## 5. Format data

In [17]:
# translate column with date to English
domestic_monthly.rename(columns={'nr_ano_mes_partida_real': 'year_month'}, inplace=True)
international_monthly.rename(columns={'nr_ano_mes_partida_real': 'year_month'}, inplace=True)

# start datasets from 2008 to remove sparse data
domestic_monthly = domestic_monthly[domestic_monthly['year_month'] >= '2008-01-01'].copy()
international_monthly = international_monthly[international_monthly['year_month'] >= '2008-01-01'].copy()

In [18]:
domestic_monthly.head()

,year_month,passengers,lag_1,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_6,exp_mean,exp_std,exp_min,exp_max
96,2008-01-01,4111527,3853064.0,3261641.0,3955423.0,3989260.0,3.544392e+06,3.683011e+06,3.821791e+06,271795.471773,3.024862e+06,546774.864443,2109035.0,4275973.0
97,2008-02-01,4060498,4111527.0,3518470.0,3594543.0,3344750.0,3.827687e+06,3.709028e+06,3.831980e+06,308129.873611,3.036064e+06,554997.459942,2109035.0,4275973.0
98,2008-03-01,4396453,4060498.0,3853064.0,3914926.0,3738048.0,4.008363e+06,3.786688e+06,3.891625e+06,331348.245136,3.046518e+06,561743.279292,2109035.0,4275973.0
99,2008-04-01,4607122,4396453.0,4111527.0,3261641.0,4275973.0,4.189493e+06,3.866942e+06,3.946492e+06,416096.690867,3.060154e+06,575102.495478,2109035.0,4396453.0
100,2008-05-01,4933469,4607122.0,4060498.0,3518470.0,4266889.0,4.354691e+06,4.091189e+06,3.974088e+06,386118.922224,3.075623e+06,592733.630836,2109035.0,4607122.0


In [19]:
international_monthly.head()

,year_month,passengers,lag_1,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_6,exp_mean,exp_std,exp_min,exp_max
96,2008-01-01,468968,412477.0,350468.0,403574.0,339707.0,370535.000000,363952.833333,346335.833333,35404.430375,416184.562500,68592.490412,253080.0,589624.0
97,2008-02-01,443856,468968.0,348660.0,343351.0,296983.0,410035.000000,374851.833333,357107.583333,54795.962674,416728.721649,68444.449348,253080.0,589624.0
98,2008-03-01,473181,443856.0,412477.0,325187.0,331584.0,441767.000000,391602.666667,369347.000000,58478.573237,417005.530612,68145.846794,253080.0,589624.0
99,2008-04-01,425615,473181.0,468968.0,350468.0,348481.0,462001.666667,416268.333333,381146.750000,56022.014704,417572.959596,68031.946427,253080.0,589624.0
100,2008-05-01,437111,425615.0,443856.0,348660.0,329109.0,447550.666667,428792.833333,387574.583333,45845.017978,417653.380000,67692.255922,253080.0,589624.0
